In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("lab4.ipynb")

![](img/575-lab-banner.png)

# Lab 4: Transformer-based image captioning

## Imports

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image
import torch
from torch import nn
from torch.utils.data import DataLoader
import math
from PIL import Image as PIL_Image
from tqdm import tqdm, trange

pd.set_option("display.max_colwidth", 200)

You may need to install the `tqdm` library in your 575 conda environment if it’s not already available. You can do this by running in the environment:

```pip install tqdm```

<br><br>

<!-- BEGIN QUESTION -->

<div class="alert-warning">

## Submission instructions <a name="si"></a>
rubric={mechanics}

You will receive marks for correctly submitting this assignment by following the instructions below:
    
- Be sure to follow the [general lab instructions](https://ubc-mds.github.io/resources_pages/general_lab_instructions/).
- [Here](https://github.com/UBC-MDS/public/tree/master/rubric) you will find the description of each rubric used in MDS.
- Make at least three commits in your lab's GitHub repository.    
- Push the final .ipynb file with your solutions to your GitHub repository for this lab.        
- Before submitting your lab, run all cells in your notebook to make sure there are no errors by doing `Kernel -> Restart Kernel and Clear All Outputs` and then `Run -> Run All Cells`. Notebooks without the output displayed may not be graded at all (because we need to see the output in order to grade your work).     
- Upload the .ipynb file to Gradescope.
- Make sure that your plots/output are rendered properly in Gradescope.    
- If the .ipynb file is too big or doesn't render on Gradescope for some reason, also upload a pdf (preferably WebPDF) or html export of .ipynb file with your solutions so that TAs can view your submission on Gradescope. 
- The data you download for this lab <b>SHOULD NOT BE PUSHED TO YOUR REPOSITORY</b> (there is also a `.gitignore` in the repo to prevent this).
- Include a clickable link to your GitHub repo for the lab just below this cell.
</div>    

_Points:_ 2

YOUR REPO LINK GOES HERE

<!-- END QUESTION -->

_This lab is a bit challenging. Start early and make use of all the resources available to you._

<br><br>

## Exercise 1: Warm up
<hr>

In this exercise, you'll implement a text-only Transformer decoder model that predicts the next token at each position in a sequence. You’ll define the model, perform a forward pass, and compute the loss — but you won't need to write a full training loop. This is a warm-up to help you become comfortable with:

- Embedding layers

- Positional encoding

- Input shape conventions in PyTorch's Transformer API

- Causal masking


First, ensure that you have `transformers` library installed in your course conda environment. 


```
>> conda activate 575
>> conda install conda-forge::transformers
```
or 

```
>> pip install transformers
```

If the above doesn't work, check out the detailed installation instructions [here](https://huggingface.co/docs/transformers/installation).

**Starter code**

The code block below sets up a small vocabulary and two example tokenized sentences related to data science and the MDS program. It performs the following:

- Defines a vocabulary with common words plus special tokens (`<PAD>` (for padding), `<SOS>` (start of sentence), `<EOS>`(end of sentence)).

- Creates mappings from words to indices (`word2idx`) and back (`idx2word`).

- Tokenizes toy sentences using these indices.

- Pads the sentences so they all have the same length, which is necessary for batching.

- Converts the padded list of sentences into a PyTorch tensor for further processing.

In [3]:
vocab = ["<PAD>", "<SOS>", "<EOS>", "data", "science", "models", "train", "well", "MDS", "students"]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(vocab)

# Toy MDS-themed tokenized sentences (you can add more!)
# <SOS> MDS students train models <EOS>
# <SOS> data science models train well <EOS>
tokenized = [
    [word2idx["<SOS>"], word2idx["MDS"], word2idx["students"], word2idx["train"], word2idx["models"], word2idx["<EOS>"]],
    [word2idx["<SOS>"], word2idx["data"], word2idx["science"], word2idx["models"], word2idx["train"], word2idx["well"], word2idx["<EOS>"]],
]

# Pad to the same length
max_len = max(len(seq) for seq in tokenized)
padded = [seq + [word2idx["<PAD>"]] * (max_len - len(seq)) for seq in tokenized]
input_tensor = torch.tensor(padded)  # (batch_size, seq_len)
print("Input tensor:\n", input_tensor)

Input tensor:
 tensor([[1, 8, 9, 6, 5, 2, 0],
        [1, 3, 4, 5, 6, 7, 2]])


Now let's prepare inputs and targets for next-token prediction. 
To train the model to predict the next token at each position, we need to shift the input sequence:

- `input_tensor` is the full input sequence fed to the decoder (e.g., starting with `<SOS>`).

- `target_output` is the expected output: it's the input sequence shifted one position to the left, with the first token removed and a `<PAD>` token appended at the end to maintain the same length.

In [4]:
# Get the PAD token ID
pad_token_id = word2idx["<PAD>"]

# Remove the first token, shape becomes (batch_size, seq_len - 1)
shifted = input_tensor[:, 1:]

# Append a PAD token at the end of each sequence
pad_column = torch.full((input_tensor.size(0), 1), pad_token_id, dtype=torch.long)

# Concatenate along the sequence dimension
target_output = torch.cat([shifted, pad_column], dim=1)

In [5]:
input_tensor

tensor([[1, 8, 9, 6, 5, 2, 0],
        [1, 3, 4, 5, 6, 7, 2]])

In [6]:
target_output

tensor([[8, 9, 6, 5, 2, 0, 0],
        [3, 4, 5, 6, 7, 2, 0]])

In [7]:
print("input_tensor shape:", input_tensor.shape) # batch_size, seq_len
print("target_output shape:", target_output.shape) # batch_size, seq_len

input_tensor shape: torch.Size([2, 7])
target_output shape: torch.Size([2, 7])


<br><br>

In [8]:
# The PositionalEncoding model is already defined for you. Do not change this class.
# We'll use this class in this exercise as well as the next exercise. 

class PositionalEncoding(nn.Module):
    """
    Implements sinusoidal positional encoding as described in "Attention is All You Need".

    Args:
        d_model (int): Dimension of the embedding space.
        dropout (float): Dropout rate after adding positional encodings.
        max_len (int): Maximum length of supported input sequences.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create a (max_len, 1) position tensor: [[0], [1], ..., [max_len-1]]
        positions = torch.arange(max_len).unsqueeze(1)

        # Compute the scaling terms for each dimension (even indices only)
        scale_factors = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        # Initialize the positional encoding matrix with shape (max_len, 1, d_model)
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(positions * scale_factors)  # Apply sine to even indices
        pe[:, 0, 1::2] = torch.cos(positions * scale_factors)  # Apply cosine to odd indices

        # Register as buffer (not a trainable parameter)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Adds positional encoding to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (seq_len, batch_size, d_model)

        Returns:
            torch.Tensor: Tensor with positional encoding added.
        """
        seq_len = x.size(0)
        x = x + self.pe[:seq_len]
        return self.dropout(x)

<br><br>

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
### 1.1 `TransformerDecoderLayer` 
rubric={reasoning}


**Your tasks:**

- Complete the `TextDecoderModel` class implementation below. 
- Run the provided test code and make sure you understand how the model output and loss are computed. 

</div>

<div class="alert alert-warning">

Solution_1_1
    
</div>

_Points:_ 8

_Type your answer here, replacing this text._

In [9]:
class TextDecoderModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # TODO 1: Define the embedding layer to convert input text tokens into vectors
        self.embedding = ...

        # TODO 2: Define positional encoding. The class `PositionalEncoding` is defined in utils.py 
        self.pos_encoding = ...

        # TODO 3: Define the Transformer decoder
        decoder_layer = ...
        self.decoder = ...
 
        # TODO 4: Final linear layer to map decoder output to vocab size
        self.output_layer = ...

    def forward(self, input_ids):
        """
        input_ids: Tensor of shape (batch_size, seq_len)
        """
        batch_size, seq_len = input_ids.shape

        # TODO 5: Embed input tokens, permute, and apply positional encoding 

        # Embed input tokens
        embeddings = ...

        # Permute to change the shape to (seq_len, batch_size, d_model), which is what nn.TransformerDecoder expects
        embeddings = ...

        # Apply positional encoding
        embeddings = ...

        # TODO 6: Create a causal mask of shape (seq_len, seq_len) 
        mask = ...
        mask = ...


        # TODO 7: Pass through the Transformer decoder. You can pass embeddings as `tgt` as well as `memory` 
        decoder_output = ...
        
        # TODO 8: Project output to vocab size. The output shape should be (seq_len, batch_size, vocab_size)
        output = ...

        return output


In [10]:
d_model = 32
n_heads = 4
num_layers = 2

seq_len = max_len
batch_size = 2

model = TextDecoderModel(vocab_size, d_model, n_heads, num_layers)
logits = model(input_tensor)  
logits.shape # Output shape: (seq_len, batch_size, vocab_size)

AttributeError: 'ellipsis' object has no attribute 'shape'

In [ ]:
assert logits.shape == (seq_len, batch_size, vocab_size)

In [ ]:
# So change shape to (batch_size, vocab_size, seq_len)
logits = logits.permute(1, 2, 0)
print("Logits shape:", logits.shape)

In [ ]:
assert logits.shape == (batch_size, vocab_size, seq_len)

In [ ]:
target_output.shape

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<PAD>"])
loss = criterion(logits, target_output)
loss.item()

<!-- END QUESTION -->

<div class="alert alert-warning">
⚠️ Don't forget to <code>git commit</code>. Regular commits will help you track your progress!  
</div>

<br><br><br><br>

## Exercise 2: Transformer-based caption generation
<hr>

Now it's time to bring everything together and build a fun and meaningful application: **caption generation**.

**Caption generation** is the task of automatically producing accurate, grammatically correct text descriptions for images. It involves identifying the key objects, actions, and context within an image, and expressing them in natural language. This task sits at the intersection of computer vision and natural language processing, using a vision model to extract features from the image, and a language model to generate the descriptive text.

The concept of caption generation was notably discussed in [Andrej Karpathy's Dissertation](https://cs.stanford.edu/people/karpathy/main.pdf), which laid the groundwork for many modern approaches. The diagram below illustrates a high-level overview: image features are first extracted using a pre-trained computer vision model (like a CNN), and then passed to a language model to generate captions word by word. 

![Captioning](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/caption-1.png "Captioning")
[source](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/caption-1.png)
<br><br>

In this exercise, you'll train your own **transformer-based caption generation model** from scratch using PyTorch! This task involves coordinating several components, and while it may feel a bit overwhelming at times, take it step by step, and be patient with yourself. We hope this hands-on experience deepens your understanding of how these models work and proves both challenging and rewarding by the end.


**Tools we will use:**

- `PyTorch` 
- [Hugging Face Transformers](https://huggingface.co/)
- [Pre-trained CNN models]()

The image captioning task consists of several key components, which we have divided into six distinct tasks. Below, we outline these tasks and provide a diagram that illustrates them. We will guide you through each task step-by-step.

- **TASK 1:** Extract image features using pre-trained vision models.
- **TASK 2:** Tokenize captions using pre-trained language models.
- **TASK 3:** Create a PyTorch dataset for training.
- **TASK 4:** Implement the `ImageCaptioning` class, which uses the transformer architecture for caption generation.
- **TASK 5:** Write the PyTorch training loop to train the model.
- **TASK 6:** Generate captions and evaluate the generated captions.


![](img/image-captioning-overview.png)

We will train the model using [the flickr8k dataset](https://www.kaggle.com/datasets/adityajn105/flickr8k). If you are working locally, download the dataset and place it in the lab directory. We recommend working on your laptop during the prototyping phase. Once you feel confident about the model, switch to Kaggle notebooks, which offer access to GPUs for up to 30 hours per week.

Follow these instruction to upload and run this lab notebook on Kaggle:

1. Go to https://www.kaggle.com/datasets/adityajn105/flickr8k.
2. Press "New Notebook" to create a new notebook
3. In the new notebook, go to "File", then "Import notebook" and upload this notebook on Kaggle.
4. On the right, go to "Session options". In "ACCELERATOR", choose a GPU you want to use.
5. You can now begin working on this lab using free GPUs 🙂. 

Ensure that you are using the correct device to fully utilize the GPU capabilities.

In [ ]:
# The folder containing the data 
image_folder = "flickr8k"

# If you run the notebook on Kaggle, you can use the following line
# image_folder = "/kaggle/input/flickr8k"

In [ ]:
# Read and show the first few lines of caption.txt 
df = pd.read_csv(f"{image_folder}/captions.txt", sep=',')
df.head()

In [ ]:
# Set the appropriate device depending upon your hardware. 

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu') 
print(device)

<br><br>

### TASK 1: Extracting image features using pre-trained vision models
rubric={accuracy}

The initial step in the image captioning process involves extracting image features. This is commonly achieved by extracting the output from the final linear layer of a Convolutional Neural Network (CNN). Instead of building our own CNN from scratch, we will use a pre-trained CNN model.

In DSCI 572, we explored how to extract features from an image using a pre-trained vision model through `torchvision.models`. In his exercise, you will learn to use the [Hugging Face transformers library](https://huggingface.co/) to preprocess images and extract image features.

![](img/task1.png)

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
### 2.1: TASK 1
rubric={accuracy}
    
**Your tasks:**

1. Complete the `ImageFeatureExtractor` class implemented below. This class uses a pre-trained `ResNet18` model to obtain image features. 

**Hints:**
- For an example of feature extraction using the ResNet model, refer to [this guide](https://huggingface.co/docs/transformers/main/en/model_doc/resnet#transformers.ResNetModel) on the Hugging Face documentation. 
- We want to use the feature after the final pooling layer ([`outputs.pooler_output`](https://huggingface.co/docs/transformers/main/en/model_doc/resnet#transformers.FlaxResNetModel.__call__)).
- Ensure that the output tensor has the shape (1, 512). You may need to reshape the tensor to achieve this.
- To improve processing speed, move the input tensors to a GPU (if available). Remember to transfer the resulting tensor back to CPU memory afterwards. Consider why this is necessary, especially when dealing with large datasets.

<div>

<div class="alert alert-warning">

Solution_2_1
    
</div>

_Points:_ 4

In [ ]:
import torch
from transformers import AutoFeatureExtractor, ResNetModel

class ImageFeatureExtractor:
    """Extracts image features using a pretrained ResNet model."""

    def __init__(self, device, model_name="microsoft/resnet-18"):
        self.device = device
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
        self.resnet_model = ResNetModel.from_pretrained(model_name).to(device)
        self.resnet_model.eval()

    def __call__(self, image) -> torch.Tensor:
        """Returns a (1, 512) tensor of features extracted from the input image."""
        # TODO: Write code to get and return image features 
        ...


In [ ]:
# Now let's open an image 
image = PIL_Image.open('{}/Images/{}'.format(image_folder, df.iloc[34396].image))
image

In [ ]:
# Your class should pass these two tests. feel free to explore the image features 
feature_extractor = ImageFeatureExtractor(device=device)
image_features = feature_extractor(image) 
assert isinstance(image_features, torch.Tensor)
assert image_features.shape == (1, 512)

<!-- END QUESTION -->

<br><br>

### TASK 2: Implement a tokenizer using pre-trained language models

Next, we will process text data by tokenizing it. We will use the [AutoTokenizer](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#autotokenizer) from Hugging Face, specifically the `bert-base-cased` model,  which you explored in the previous lab. Given that the vocabulary of `bert-base-cased` is much larger compared to that of our captions dataset, we'll implement a wrapper class for `AutoTokenizer` to provide custom token-to-vocabulary mappings.

![](img/task2.png)

<!-- BEGIN QUESTION -->

<div class="alert alert-info">

### Exercise 2.2: TASK 2
rubric={accuracy}

**Your tasks:**
1. Read through the code and try to understand the purpose of each method in the class.
      
2. Complete the `decode()` method of the `TokenizerWrapper` class implemented below:
    - Use `tokenizer.decode` to convert a list of token IDs back into a text string.
    
3. Run the test code to examine the decoded caption. 
</div>

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Points:_ 2

In [ ]:
from transformers import AutoTokenizer
from tqdm import trange

class TokenizerWrapper:
    """Wraps AutoTokenizer with a custom vocabulary mapping."""

    def __init__(self, model_name="bert-base-cased"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Initialize mappings with special tokens: [PAD] -> 0, [CLS] -> 1, [SEP] -> 2
        self.token_id_to_vocab_id = {0: 0, 101: 1, 102: 2}
        self.vocab_id_to_token_id = {0: 0, 1: 101, 2: 102}
        
        self.vocab_id = 3  # Start after special tokens
        self.padding_len = None

    def build_dictionary(self, captions: list[str]):
        """Builds vocabulary from a list of captions and sets padding length."""
        tokenized = self.tokenizer(captions, padding='longest').input_ids
        self.padding_len = len(tokenized[0])

        for tokens in tokenized:
            for token_id in tokens:
                if token_id not in self.token_id_to_vocab_id:
                    self.token_id_to_vocab_id[token_id] = self.vocab_id
                    self.vocab_id_to_token_id[self.vocab_id] = token_id
                    self.vocab_id += 1

    def get_vocab_size(self) -> int:
        """Returns the size of the custom vocabulary."""
        assert len(self.token_id_to_vocab_id) == len(self.vocab_id_to_token_id)
        return self.vocab_id

    def tokenize(self, text: str) -> list[int]:
        """Tokenizes text using custom vocabulary (requires build_dictionary first)."""
        assert self.padding_len is not None, "Call build_dictionary() before tokenizing."
        token_ids = self.tokenizer(text, padding='max_length', max_length=self.padding_len).input_ids
        return [self.token_id_to_vocab_id[token_id] for token_id in token_ids]

    def decode(self, vocab_ids: list[int]) -> str:
        """Decodes a list of custom vocab IDs into a string."""
        token_ids = [self.vocab_id_to_token_id[vocab_id] for vocab_id in vocab_ids]
        # TODO: Use `self.tokenizer.decode` to convert a list of token IDs back into a text string.
        text = ...
        return text.replace('[CLS] ', '').replace(' [SEP]', '').replace(' [PAD]', '')


In [ ]:
# Build the dictionary for our tokenizer  
tokenizer_wrapper = TokenizerWrapper()
tokenizer_wrapper.build_dictionary(df["caption"].to_list())

In [ ]:
# What's the size of our custom vocabulary
tokenizer_wrapper.get_vocab_size() 

In [ ]:
# let's try to tokenize the caption corresponding to the image we saw in the last task, 
# and decode the tokens back to the caption

caption_tokens = tokenizer_wrapper.tokenize(df.iloc[34396].caption)
decoeded_caption = tokenizer_wrapper.decode(caption_tokens)
print('Caption:', df.iloc[34396].caption)
print('Tokens:', caption_tokens)
print('Decoded caption:', decoeded_caption)

# Your Caption and the Decoded caption should match here. 

# your class should pass these two tests. feel free to explore the image features 
assert isinstance(caption_tokens, list)
assert isinstance(decoeded_caption, str)
assert len(caption_tokens) == tokenizer_wrapper.padding_len

<!-- END QUESTION -->

<br><br>

### TASK 3: Data splitting and creating a Pytorch dataset

So far, we have implemented an image feature extractor to extract feature vectors from given images and a tokenizer to obtain encoded representations of captions. Now, let's prepare our data for both training and evaluating our models.

First, we'll split the data into training and testing subsets.

In [ ]:
import pandas as pd
import numpy as np

def train_test_split_by_image(data_df, sample_size=None, train_ratio=0.8, seed=100):
    """
    Splits the dataframe into training and testing datasets based on unique images.
    
    Parameters:
        data_df (pandas.DataFrame): The dataset to split, containing at least 'image' and 'caption' columns.
        sample_size (int): The number of samples to consider. Useful during prototyping.
        train_ratio (float): The proportion of the dataset to allocate to the training set.
        seed (int): Seed for random number generator for reproducibility.

    Returns:
        train_df (pandas.DataFrame): Training dataset.
        test_df (pandas.DataFrame): Testing dataset.
    """
    np.random.seed(seed)
    unique_images = np.random.permutation(data_df['image'].unique())

    if sample_size:
        unique_images = unique_images[:sample_size]

    split_point = int(len(unique_images) * train_ratio)
    train_images, test_images = unique_images[:split_point], unique_images[split_point:]

    train_df = data_df[data_df['image'].isin(train_images)]
    test_df = data_df[data_df['image'].isin(test_images)]

    return train_df, test_df

In [ ]:
# For prototyping 
train_df, test_df = train_test_split_by_image(df, sample_size = 100, train_ratio=0.8)
print(f'The number of rows in the training set is {train_df.shape[0]} and the number of unique images is {int(train_df.shape[0]/5)}')
print(f'The number of rows in the test set is {test_df.shape[0]} and the number of unique images is {int(test_df.shape[0]/5)}')
train_df.head()

In [ ]:
# # once you are confident about your model, use the entire data!!! 
# train_df, test_df = train_test_split_by_image(df, train_ratio=0.8)
# print(f'The number of rows in the training set is {train_df.shape[0]} and the number of unique images is {train_df.shape[0]/5}')
# print(f'The number of rows in the test set is {test_df.shape[0]} and the number of unique images is {test_df.shape[0]/5}')
# train_df.head()

Now that we have train and test datasets, the next steps involve:

- Processing each image and caption in the training and testing sets.
- Creating a PyTorch dataset for efficient batch loading and processing.
  
We will first write a function called `build_data` to convert the data into a torch tensor, and then write a class called `PytorchDataset` to create a PyTorch dataset.
<br><br>

<!-- BEGIN QUESTION -->

### 2.3: TASK 3
rubric={accuracy}

<div class="alert alert-info">
    
**Your tasks:**


1. Complete the `build_data` function below:
   
    - Use the `feature_extractor` from Exercise 2.1 to obtain the image feature.
      
    - Use the `tokenizer_wrapper` from Exercise 2.2 to convert captions into numerical (token ID) representations. Make sure to convert the list of token IDs into a PyTorch tensor using `torch.tensor`.      
<div>

<div class="alert alert-warning">

Solution_2_3
    
</div>

_Points:_ 2

_Type your answer here, replacing this text._

In [ ]:
def build_data(data_df, tokenizer_wrapper, feature_extractor, image_folder):
    """
    Constructs a dataset list from provided image and caption data.

    This function processes each entry in a dataframe, extracts image features using a specified
    feature extractor, tokenizes captions using a tokenizer wrapper, and combines these elements into
    a list where each item is a dictionary containing image features and tokenized caption data.

    Parameters:
        data_df (pandas.DataFrame): A dataframe containing at least two columns: 'image' and 'caption'.
            - 'image' column contains filenames of images.
            - 'caption' column contains the corresponding captions for the images.
        tokenizer_wrapper (TokenizerWrapper): An object encapsulating a tokenizer that provides
            a method 'tokenize' to convert text captions into token ids.
        feature_extractor (Callable): A function or callable object that accepts an image object and
            returns a tensor representing extracted features from the image.
        image_folder (str): The path to the folder where images are stored. The path should not
            end with a slash. It is assumed that images are stored within an 'Images' subfolder.

    Returns:
        list: A list of dictionaries, where each dictionary has two keys:
            - 'image': a tensor containing features of the image.
            - 'token': a tensor of token ids derived from the caption.

    Example:
        Assuming 'data_df' has two columns 'image' and 'caption' where 'image' contains filenames:
        >>> data_df = pd.DataFrame({
        ...     'image': ['image1.jpg', 'image2.jpg'],
        ...     'caption': ['A cat on a mat.', 'A dog in the fog.']
        ... })
        >>> dataset = build_data(data_df, tokenizer_wrapper, feature_extractor, '/path/to/images')
        >>> print(dataset[0]['image'])  # Outputs the tensor of image features for 'image1.jpg'
        >>> print(dataset[0]['token'])  # Outputs the tensor of token ids for 'A cat on a mat.'
    """    

    dataset = []
    for i in trange(len(data_df)):
        row = data_df.iloc[i]
        image_path = f"{image_folder}/Images/{row.image}"
        image = PIL_Image.open(image_path)

        # TODO: Get image features 
        image_features = ...

        # TODO: Get caption tokens
        caption_tokens = ...
        
        dataset.append({'image': image_features, 'token': caption_tokens})
    return dataset
    

In [ ]:
train_data = build_data(train_df, tokenizer_wrapper, feature_extractor, image_folder)
test_data = build_data(test_df, tokenizer_wrapper, feature_extractor, image_folder)

In [ ]:
# Get the dimension of the image feature
image_dim = train_data[0]['image'].shape[-1]
vocab_size = tokenizer_wrapper.get_vocab_size()
print(f'Image feature dimension is {image_dim}. The vocab size is {vocab_size}.')

# your image_dim should pass this test
assert isinstance(image_dim, int)

In [ ]:
class PytorchDataset:
    """
    A PyTorch-compatible dataset that returns:
    - the original token sequence,
    - image features,
    - and the target sequence (input shifted left with padding at the end).
    """

    def __init__(self, data, pad_vocab_id=0):
        self.data = data
        self.pad_token = torch.tensor([pad_vocab_id])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens = self.data[idx]['token']
        image = self.data[idx]['image']
        target = torch.cat([tokens[1:], self.pad_token])
        return tokens, image, target


In [ ]:
train_dataset = PytorchDataset(train_data)
test_dataset = PytorchDataset(test_data)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=50, shuffle=False)

In [ ]:
# Now let's get the first data from PytorchDataset
train_token, train_image, train_target = train_dataset[0]

# Your dataset should pass these tests
assert train_token.shape == train_target.shape
assert torch.all(train_token[1:] == train_target[:-1]).item()

In [ ]:
# Shape of the image features
train_image.shape

In [ ]:
# Numericalized caption
train_token 

In [ ]:
# decoded caption
tokenizer_wrapper.decode(train_token.tolist()) 

In [ ]:
# the target of the train_token (i.e., training example) 
train_target

In [ ]:
# decoded caption
tokenizer_wrapper.decode(train_target.tolist()) 

In [ ]:
# Now let's get a batch of data from DataLoader
train_text, train_image, train_target = next(iter(train_dataloader))
train_text = train_text.to(device)
train_image = train_image.to(device)

assert train_image.ndim == 3
assert train_text.ndim == 2

In [ ]:
train_text.shape # batch_size, max_len

In [ ]:
train_image.shape # batch_size, max_len, img_size

In [ ]:
train_target.shape # batch_size, max_len

<!-- END QUESTION -->

<br><br>

### TASK 4: Implement your own image caption model using PyTorch
<hr>

Now we are ready to define our own transformer model tailored for image captioning! 

![](img/task4.png)


<br><br>

The diagram below shows how image features and token sequences are processed in a caption generation model.

- Image features are projected into the model's embedding space using `self.image_embedding`, then reshaped to act as the decoder's memory.

- Token IDs are embedded using `self.text_embedding` and enriched with positional information using `self.pos_encoding` (defined in `utils.py`).

- These two inputs are passed to the `TransformerDecoder`, which produces context-aware token representations.

- The decoder output is passed through a `linear_layer` to generate vocabulary logits, followed by a softmax to produce token probabilities.

- During inference, a token is sampled from this distribution and used to generate the next word.

![](img/transformer-decoder.png)

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
### 2.4: TASK 4

rubric={accuracy}

**Your tasks:**

Complete the `forward` method of the `ImageCaptionModel` class implemented below:   

**Hints:**
- Obtain the image embedding by passing the image feature through the corresponding layer. Make sure to permute the dimensions to reshape the encoded image feature to have the shape (`seq_len`, `batch_size`, `d_model`). Note that in this caption generation setup, we are assuming that the image features represent a single "token", so the `seq_len` for the image is 1.
          
- Process the token IDs to retrieve the token embedding. Ensure to 
    - Scale the caption embedding by `math.sqrt(self.embedding_size)`, similar to how we did it in Exercise 1.
 
    - Permute the dimensions to ensure the encoded captions have the shape (`seq_len`, `batch_size`, `d_model`).
 
    - Add positional encoding to the sequences.
          
    - Pass all variables (image embeddings, caption embeddings, and masks) to the transformer model and obtain the output from the final linear layer.
  
</div>

<div class="alert alert-warning">

Solution_2_4
    
</div>

_Points:_ 15

In [ ]:
class ImageCaptionModel(nn.Module):
    def __init__(self, d_model, n_heads, num_layers, image_dim, vocab_size, device, dropout=0.1):
        """
        Initialize the ImageCaptionModel which uses a transformer decoder architecture
        for generating image captions.

        Parameters:
            d_model (int): The number of expected features in the encoder/decoder inputs.
            n_heads (int): The number of heads in the multiheadattention models.
            num_layers (int): The number of sub-decoder-layers in the transformer.
            image_dim (int): The dimensionality of the input image features.
            vocab_size (int): The size of the vocabulary.
            device (torch.device): The device on which the model will be trained.
            dropout (float): The dropout value used in PositionalEncoding and TransformerDecoderLayer.
        """        
        super(ImageCaptionModel, self).__init__()
        self.d_model = d_model
        self.device = device
        # Positional Encoding to add position information to input embeddings
        self.pos_encoding = PositionalEncoding(d_model=d_model, dropout=dropout)

        # Here is our transformer architecture
        
        # Transformer Decoder to generate captions from image features and text inputs
        self.TransformerDecoder = nn.TransformerDecoder(
            decoder_layer=nn.TransformerDecoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout), 
            num_layers=num_layers
        )

        # Linear layer to convert image features to a dimensionality that matches the model's
        self.image_embedding = nn.Linear(image_dim , d_model)

        # Embedding layer for converting input text tokens into vectors
        self.text_embedding = nn.Embedding(vocab_size , d_model)

        # Final linear layer to map the output of the transformer decoder to vocabulary size        
        self.linear_layer = nn.Linear(d_model, vocab_size)

        # Initialize the weights of the model
        self.init_weights()
        
    def init_weights(self):
        """
        Initialize weights of the model to small random values.
        """
        initrange = 0.1
        self.text_embedding.weight.data.uniform_(-initrange, initrange)
        self.image_embedding.weight.data.uniform_(-initrange, initrange)
        self.linear_layer.bias.data.zero_()
        self.linear_layer.weight.data.uniform_(-initrange, initrange)
        
    def forward(self, image, text):
        """
        Defines the forward pass of the model.

        Parameters:
            image (Tensor): The tensor containing image features.
            text (Tensor): The tensor containing input text tokens.

        Returns:
            Tensor: The output tensor containing the logit scores for each vocabulary token.
        """        
        
        # TODO: Use `self.image_embedding` to project the image features to the model's expected dimensionality (`d_model`)
        encoded_image = ...

        # TODO: Permute `encoded_image` to shape (seq_len, batch_size, d_model), as expected by the Transformer decoder
        encoded_image = ...

        # TODO: Convert text tokens into embeddings using `self.text_embedding`, and scale by sqrt(d_model)
        encoded_text = ...

        # TODO: Permute `encoded_text` to shape (seq_len, batch_size, d_model), as expected by the Transformer decoder
        encoded_text = ...
        
        # TODO: Apply positional encoding on the permuted text using `self.pos_encoding`        
        encoded_text = ...
        
        # TODO: Get the length of the target sequence (`encoded_text`) to create a causal mask
        seq_len = ...

        # TODO: Create a causal mask to prevent the decoder from attending to future tokens
        causal_mask = ...

        # TODO: Convert the boolean mask to float and apply -inf/0 values as required by the Transformer
        causal_mask = ...

        # TODO: Move the causal mask to the correct device (`self.device`)
        causal_mask = ...

        # TODO: Pass the embeddings and memory to the Transformer decoder
        transformer_output = ...

        # TODO: Apply the linear layer to project decoder outputs to vocabulary logits
        final_layer_output = ...
        
        return final_layer_output


In [ ]:
# Now let's try your model. 
# Define the hyperparameters and initalize the model. Feel free to change these hyperparameters. 
d_model = 256 
n_heads = 4
num_layers = 8
model = ImageCaptionModel(d_model=d_model, n_heads=n_heads, num_layers=num_layers, image_dim=image_dim, vocab_size=vocab_size, device=device).to(device)

In [ ]:
# pass inputs to your model
output = model(train_image, train_text)

# output should pass the test
assert output.shape == (train_text.shape[1], train_text.shape[0], vocab_size)

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
### 2.5 TASK 5: Implement the training loop
rubric={accuracy}

Now that you've built your `ImageCaptionModel` and prepared the dataset, the next step is to train the model. Below, we've already defined the optimizer and loss function for you.


**Your tasks:**

Complete the training function below. In particular, 

1. Write code to compute the loss for the sampled training batch:
   - **Hint:** Use the criterion to calculate the loss for the training batch.
     
   - **Hint:** You might need to permute the dimensions of the model output to match the expected input shape for `torch.nn.CrossEntropyLoss`, which is (batch size, vocab size, sequence length). Refer to the [PyTorch documentation on `CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) for more details.
     
2. Similarly, write code to calculate the loss for the testing batch using the criterion.
   
3. Train your model. Consider saving it for further use.

</div>

In [ ]:
# Define the optimizer and the loss function. Feel free to change the hyperparameters. 

num_epoch = 5
clip_norm = 1.0
lr = 5e-5

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss(ignore_index=0) # Ignore the padding index

<div class="alert alert-warning">

Solution_2_5
    
</div>

_Points:_ 8

In [ ]:
def trainer(model, criterion, optimizer, train_dataloader, test_dataloader, epochs=5, patience=5):
    """
    Trains and evaluates a model using the specified dataloaders, optimizer, and loss criterion.
    Implements early stopping if the test loss increases for `patience` consecutive epochs.

    Parameters:
        model (nn.Module): The model to train.
        criterion (loss function): The loss function, e.g., CrossEntropyLoss.
        optimizer (torch.optim.Optimizer): The optimizer for training.
        train_dataloader (DataLoader): Dataloader for training data.
        test_dataloader (DataLoader): Dataloader for test/validation data.
        epochs (int): Maximum number of training epochs.
        patience (int): Number of allowed consecutive test loss increases before early stopping.

    Returns:
        train_losses (list of float): Average training loss per epoch.
        test_losses (list of float): Average test loss per epoch.
    """    
    train_losses = []
    test_losses = []
    consec_increases = 0  # For tracking how many times test loss increased
    
    for epoch in range(epochs):
        sum_train_loss = 0
        sum_test_loss = 0
        num_train_loss = 0        
        num_test_loss = 0

        # Set model to training mode (important for layers like dropout or batchnorm)
        model.train()

        # Iterate over training batches
        for train_text, train_image, target_seq in train_dataloader:

            # Move inputs and targets to the correct device
            train_text = train_text.to(device)
            train_image = train_image.to(device)
            target_seq = target_seq.to(device)

            # TODO: Forward pass — get the model's predictions # (seq_len, batch_size, vocab_size) 
            output = ...

            # TODO: Permute the output so its shape is (batch_size, vocab_size, seq_len) for CrossEntropyLoss
            output = ...

            # TODO: Compute training loss using the criterion 
            train_loss = ...

            # Backpropagation and optimization step
            optimizer.zero_grad()        
            train_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)  # Gradient clipping (prevents exploding gradients)
            optimizer.step()

            # Keep track of training loss
            sum_train_loss += train_loss.item()
            num_train_loss += 1 

        # Switch to evaluation mode (e.g., disables dropout)
        model.eval()
        with torch.no_grad():
            for test_text, test_image, target_seq in test_dataloader:

                # Move inputs and targets to the correct device
                test_text = test_text.to(device)
                test_image = test_image.to(device)
                target_seq = target_seq.to(device)

                # TODO: Forward pass for test data
                output = ...

                # TODO: Permute output shape to match loss function expectations
                output = ...

                # TODO: Compute test loss
                test_loss = ...

                # Keep track of test loss
                sum_test_loss += test_loss.item()
                num_test_loss += 1

        # Calculate average losses for the epoch
        avg_train_loss = sum_train_loss / num_train_loss
        avg_test_loss = sum_test_loss / num_test_loss
        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)

        # Print progress
        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.2f}, Test Loss = {avg_test_loss:.2f}")

        # TODO: Early stopping — stop training if test loss keeps increasing
        if epoch > 0 and test_losses[-1] > test_losses[-2]:
            consec_increases += 1
        else:
            consec_increases = 0
        if consec_increases == patience:
            print(f"Stopped early at epoch {epoch + 1} — test loss increased for {consec_increases} consecutive epochs.")
            break

    return train_losses, test_losses


In [ ]:
train_losses, test_losses = trainer(model, criterion, optimizer,train_dataloader, test_dataloader, epochs= num_epoch)

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## 2.6 TASK 6: Generate captions using the trained model
rubric={accuracy}

You've trained your model — now it's time to put it to the test! In this final task, you'll use your `ImageCaptionModel` to generate captions for images the model hasn't seen before.

The `display_img_captions` function (provided below) displays each image along with its original (ground truth) and generated captions. It samples at least `n_samples` images from the test set.


**Your tasks:**

1. Use the `display_img_captions` function to display the original and generated captions for at least 10 images from the test set. Feel free to modify the function to suit your needs (e.g., display more examples or adjust formatting).

**Troubleshooting Tips:**
- If your model generates nonsensical captions, consider adjusting the hyperparameters. For instance, you might need to use more data or run it for more epochs. However, do not expect very high-quality captions, especially if you are using a small subset for training. The primary objective of this exercise is to familiarize yourself with the process of developing your own transformer-based model. If you have a working model, that goal is achieved, and we consider it a success!

</div>

In [ ]:
import random 
random.seed(10)

def get_test_data(index):
    """
    Retrieve text and image tensors from the test dataset at the specified index.

    Parameters:
        index (int): Index of the sample to retrieve.

    Returns:
        tuple: (test_text, test_image)
    """
    test_text, test_image, _ = test_dataset[index]
    return test_text, test_image


def generate_caption(model, image, device, max_caption_length=100, start_vocab=1, end_vocab=2):
    """
    Generate a caption for the given image using the trained model.

    Parameters:
        model (nn.Module): Trained captioning model.
        image (Tensor): Image tensor.
        device (torch.device): Device for computation.
        max_caption_length (int): Maximum number of tokens to generate.
        start_vocab (int): Vocabulary index for <START> token.
        end_vocab (int): Vocabulary index for <END> token.

    Returns:
        np.ndarray: Generated caption as a sequence of token IDs.
    """
    image = image.unsqueeze(0).to(device)
    context = torch.tensor([[start_vocab]], device=device)

    for _ in range(max_caption_length):
        logits = model(image, context)[-1]  # Get logits for last token
        probs = torch.softmax(logits, dim=-1).flatten(start_dim=1)
        next_token = torch.multinomial(probs, num_samples=1)
        context = torch.cat([context, next_token], dim=1)

        if next_token.item() == end_vocab:
            break

    return context.squeeze().cpu().numpy()

def display_img_captions(image_folder, n_samples=5, indices=None):
    """
    Display generated and actual captions for a sample of test images.

    Parameters:
        image_folder (str): Path to the folder containing image files.
        n_samples (int): Number of test samples to display.
        indices (list[int], optional): Specific test indices to use.
    """
    desired_size = (200, 200)

    if indices is None:
        indices = random.sample(range(len(test_df)), n_samples)
        print(f"Randomly selected indices: {indices}")

    for idx in indices:
        test_text, test_image = get_test_data(idx)

        # Generate caption
        generated_ids = generate_caption(model, test_image, device)
        generated_caption = tokenizer_wrapper.decode(generated_ids)
        gold_caption = tokenizer_wrapper.decode(test_text.numpy())

        # Load and resize image
        image_path = f"{image_folder}/Images/{test_df.iloc[idx]['image']}"
        image = PIL_Image.open(image_path).resize(desired_size)

        # Display image and captions
        plt.figure(figsize=(5, 5))
        plt.imshow(image)
        plt.axis('off')
        plt.title(f"Generated: {generated_caption}\nActual: {gold_caption}")
        plt.show()

        print('\n\n')


<div class="alert alert-warning">

Solution_2_6
    
</div>

_Points:_ 2

In [ ]:
...

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## 2.7 Discussion
rubric={reasoning}

**Your Tasks:**

1. Comment on the quality of the generated captions. To what extent is the model able to identify key actions and objects in the images? 
2. Discuss potential modifications in model architecture, training dataset, or hyperparameter tuning to improve performance of your image captioning model.
</div>

<div class="alert alert-warning">

Solution_2_7
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<div class="alert alert-warning">
⚠️ Don't forget to <code>git commit</code>. Regular commits will help you track your progress!  
</div>

<br><br><br><br>

<br><br>

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Exercise 3: Your takeaway and reflections
rubric={reasoning}


**Your tasks:**

- Reflect on your experience in this course and across the machine learning courses in the program. What are your three biggest takeaways or insights?

- Think about the course content across these ML courses. Which topics did you find most useful? Which were least useful? Are there any topics you wish had been covered but weren't? (I know we need to add more topics covering LLM applications.)

- Many of you may have used large language models (LLMs) like ChatGPT during your learning. How do you think LLMs could be meaningfully and effectively integrated into our machine learning curriculum?

> Please be thoughtful, thorough, and specific in your responses. We greatly value your insights.

<div>

<div class="alert alert-warning">

Solution_3
    
</div>

_Points:_ 3

<!-- END QUESTION -->

<div class="alert alert-warning">
⚠️ Don't forget to <code>git commit</code>. Regular commits will help you track your progress!  
</div>

<br><br><br><br>

Since this is your final lab and Exercise 2 was quite challenging, we will not have our "Food for thought" section in this lab. 

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

That's it! Congrats on finishing the last lab of your last machine learning course! Woohoo!! 🎉

In [ ]:
from IPython.display import Image

Image("img/eva-congrats.png")